### This is the other model of house_price_forecast, using Logarithm Regression to forecast the price

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import math
np.random.seed(0)

In [2]:
test = pd.read_csv('D:/mythings/Dataset/HousePrice/Housing Prices Competition for Kaggle Learn Users/test.csv')
train = pd.read_csv('D:/mythings/Dataset/HousePrice/Housing Prices Competition for Kaggle Learn Users/train.csv')

In [3]:
LotFrontage_avg_train = train['LotFrontage'].mean()
LotFrontage_avg_test = test['LotFrontage'].mean()
train['LotFrontage'] = train['LotFrontage'].fillna(LotFrontage_avg_train)
test['LotFrontage'] = test['LotFrontage'].fillna(LotFrontage_avg_test)
train.loc[948,'BsmtExposure'] = 'No'
test.loc[27,'BsmtExposure'] = 'No'
test.loc[888,'BsmtExposure'] = 'No'
#TotalBsmtSF
TotalBsmt_avg_train = train['TotalBsmtSF'].mean()
TotalBsmt_avg_test = test['TotalBsmtSF'].mean()
train['TotalBsmtSF'] = train['TotalBsmtSF'].fillna(TotalBsmt_avg_train)
test['TotalBsmtSF'] = test['TotalBsmtSF'].fillna(TotalBsmt_avg_test)

In [4]:
category_features = ['MSZoning','LandContour','Utilities','LotConfig','Neighborhood','Condition1',
                     'Condition2','BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st','Exterior2nd','MasVnrType'
                     ,'Foundation','Heating','Electrical','Functional','GarageType','PavedDrive','PavedDrive',
                     'Fence','MiscFeature','SaleType','SaleCondition']
encoding_features = ['Street', 'Alley','CentralAir']
ordinal_features_15 = ['ExterQual',
    #'ExterCond',
    'BsmtQual',
    #'BsmtCond','HeatingQC','KitchenQual',
    'FireplaceQu',
    'GarageQual',
    #'GarageCond'
    ]
ordinal_features_05 = ['BsmtFinType1','BsmtFinType2',]
ordinal_feature_02 =['GarageFinish']
ordinal_features_03 = ['LotShape','BsmtExposure',]
ordinal_features_25 = ['PoolQC']

In [5]:
convert_scale_15 = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}
convert_scale_05 = {'Unf': 0, 'LwQ': 1, 'Rec': 2, 'BLQ': 3, 'ALQ': 4, 'GLQ': 5}
convert_scale_LotShape = {'Reg': 0, 'IR1': 1,'IR2': 2,'IR3': 3}
convert_scale_Finish = {'Fin':3,'RFn':2,'Unf':1}

In [6]:
train[ordinal_features_15] = train[ordinal_features_15].replace(convert_scale_15)
train[ordinal_features_05] = train[ordinal_features_05].replace(convert_scale_05)
train['GarageFinish'] = train['GarageFinish'].replace(convert_scale_Finish)
train['LotShape'] = train['LotShape'].replace(convert_scale_LotShape)
train['PoolQC'] = train['PoolQC'].replace(convert_scale_15)

C:\Users\Rice\AppData\Local\Temp\ipykernel_23024\324370086.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[ordinal_features_15] = train[ordinal_features_15].replace(convert_scale_15)
C:\Users\Rice\AppData\Local\Temp\ipykernel_23024\324370086.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[ordinal_features_05] = train[ordinal_features_05].replace(convert_scale_05)
C:\Users\Rice\AppData\Local\Temp\ipykernel_23024\324370086.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in

In [ ]:
test[ordinal_features_15] = test[ordinal_features_15].replace(convert_scale_15)
test[ordinal_features_05] = test[ordinal_features_05].replace(convert_scale_05)
test['GarageFinish'] = test['GarageFinish'].replace(convert_scale_Finish)
test['LotShape'] = test['LotShape'].replace(convert_scale_LotShape)
test['PoolQC'] = test['PoolQC'].replace(convert_scale_15)

C:\Users\Rice\AppData\Local\Temp\ipykernel_23024\1152810533.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test[ordinal_features_15] = test[ordinal_features_15].replace(convert_scale_15)
C:\Users\Rice\AppData\Local\Temp\ipykernel_23024\1152810533.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test[ordinal_features_05] = test[ordinal_features_05].replace(convert_scale_05)
C:\Users\Rice\AppData\Local\Temp\ipykernel_23024\1152810533.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in 

In [8]:
Area = ['LotArea', 'LotFrontage','TotalBsmtSF','GrLivArea','LotFrontage', '1stFlrSF']

In [9]:
def detect_outliers_iqr(df, feature):
    df[feature] = sorted(df[feature])
    q1 = np.percentile(df[feature], 25)
    q3 = np.percentile(df[feature], 75)
    IQR = q3 - q1
    lower_bound =q1 - (1.5 *IQR)
    upper_bound = q3 + (1.5*IQR)
    return lower_bound, upper_bound

In [10]:
#Detect and remove outliers in Train Dataset
## LotArea
#lower_bound, upper_bound = detect_outliers_iqr(train, 'LotArea')
#train = train[(train['LotArea'] > lower_bound) & (train['LotArea'] < upper_bound)]
## GrLivArea
#lower_bound, upper_bound = detect_outliers_iqr(train, 'GrLivArea')
#train = train[(train['GrLivArea'] > lower_bound) & (train['GrLivArea'] < upper_bound)]
#TotRmsAbvGr
train = train[(train['TotRmsAbvGrd'] > 2) & (train['TotRmsAbvGrd'] < 14)]

In [11]:
def log_transformation(df, features, features_log):
    df[features] = df[features].replace(0, None)
    df[features] = df[features].astype(float)
    df[features_log] = np.log10(1 + df[features])
    df.drop(columns = features , axis = 1 , inplace = True )
    return df

In [12]:
features_train = ['LotArea','LotFrontage','1stFlrSF','GrLivArea','SalePrice']
features_log_train = []
for col in features_train:
    features_log_train.append(col +'_log')

In [13]:
features_test =['LotArea','LotFrontage','1stFlrSF','GrLivArea',]
features_log_test = []
for col in features_test:
    features_log_test.append(col +'_log')

In [ ]:
log_transformation(train,features_train, features_log_train)
log_transformation(test,features_test, features_log_test)

,Id,MSSubClass,MSZoning,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,...,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,LotArea_log,LotFrontage_log,1stFlrSF_log,GrLivArea_log
0,1461,20,RH,Pave,NaN,0,Lvl,AllPub,Inside,Gtl,...,NaN,0,6,2010,WD,Normal,4.065318,1.908485,2.952792,2.952792
1,1462,20,RL,Pave,NaN,1,Lvl,AllPub,Corner,Gtl,...,Gar2,12500,6,2010,WD,Normal,4.154363,1.913814,3.123852,3.123852
2,1463,60,RL,Pave,NaN,1,Lvl,AllPub,Inside,Gtl,...,NaN,0,3,2010,WD,Normal,4.140854,1.875061,2.968016,3.212188
3,1464,60,RL,Pave,NaN,1,Lvl,AllPub,Inside,Gtl,...,NaN,0,6,2010,WD,Normal,3.999087,1.897627,2.967080,3.205475
4,1465,120,RL,Pave,NaN,1,HLS,AllPub,Inside,Gtl,...,NaN,0,1,2010,WD,Normal,3.699491,1.643453,3.107549,3.107549
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2915,160,RM,Pave,NaN,0,Lvl,AllPub,Inside,Gtl,...,NaN,0,6,2006,WD,Normal,3.287130,1.342423,2.737987,3.038620
1455,2916,160,RM,Pave,NaN,0,Lvl,AllPub,Inside,Gtl,...,NaN,0,4,2006,WD,Abnorml,3.277609,1.342423,2.737987,3.038620
1456,2917,20,RL,Pave,NaN,0,Lvl,AllPub,Inside,Gtl,...,NaN,0,9,2006,WD,Abnorml,4.301052,2.206826,3.088136,3.088136
1457,2918,85,RL,Pave,NaN,0,Lvl,AllPub,Inside,Gtl,...,Shed,700,7,2006,WD,Normal,4.018784,1.799341,2.987219,2.987219


In [15]:
#encoder = OneHotEncoder()
#for feature in ['Street','Alley','LandSlope']:
#    train[feature] = encoder.fit_transform(train[feature])
#    test[feature] = encoder.fit_transform(test[feature])
train = pd.get_dummies(train, columns=['Street','Alley','LandSlope'], drop_first=True)
test = pd.get_dummies(test, columns = ['Street','Alley','LandSlope'], drop_first=True)

In [16]:
features = ['GrLivArea_log','LotArea_log','1stFlrSF_log']

In [17]:
indie_vars = features + ordinal_features_15+ ordinal_features_05+ \
['TotRmsAbvGrd','FullBath'] +['Street_Pave','Alley_Pave','LandSlope_Mod','LandSlope_Sev'] +\
['MSSubClass','YearBuilt'] 

In [18]:
X = train[indie_vars]
y = train['SalePrice_log']
train_X, test_X, train_y, test_y = train_test_split(X,y, test_size= 0.2,random_state = 0)

In [19]:
house_model = RandomForestRegressor(random_state= 0)
house_model.fit(X,y)
house_predictions = house_model.predict(test_X)
house_model_r2 = r2_score(test_y, house_predictions)
print('Training Score: {}'.format(house_model.score(train_X, train_y)))
print('Testing Score: {}'.format(house_model.score(test_X, test_y)))

Training Score: 0.9737938115003587
Testing Score: 0.9827760260375744


In [20]:
print('The predictions of Price: {}'.format(house_predictions[0:5]))
print('RMSE: {}'.format(root_mean_squared_error(10**test_y,10**house_predictions)))

The predictions of Price: [5.14774816 5.25629092 4.95288711 5.1495004  5.50750995]
RMSE: 11896.519285460983


In [21]:
#Remove 'Street','Alley', RMSE: 0.024466077592765815
# Adding 'FullBath' + Removing 'Street' , 'Alley', RSME: 0.024463973162251698
# Detect and remove outliers in 'TotRmsAbvGrd' + Adding 'FullBath' + Removing 'Street','Alley',
## RMSE = 0.02370472816966739
# Detect and remove outliers in 'TotRmsAbvGrd' + Adding 'FullBath' + OneHotEncoder 'Street','Alley'
## RMSE = 0.023618514535610886
# Detect and remove outliers in 'TotRmsAbvGrd' + Adding 'FullBath' + OneHotEncoder 'Street','Alley','LandSlope'
'''indie_vars = features + ordinal_features_15+ ordinal_features_05+
    ['TotRmsAbvGrd','FullBath'] +['Street_Pave','Alley_Pave','LandSlope_2','LandSlope_3',] +
    ['MSSubClass','YearBuilt']'''
## RMSE = 0.02352149276442018


"indie_vars = features + ordinal_features_15+ ordinal_features_05+\n    ['TotRmsAbvGrd','FullBath'] +['Street_Pave','Alley_Pave','LandSlope_2','LandSlope_3',] +\n    ['MSSubClass','YearBuilt']"

In [22]:
output2 = pd.DataFrame({'Id': test['Id'], 
'SalePrice': 10 ** (house_model.predict(test[indie_vars]))})
#output2.to_csv('submission_log02.csv', index = False)

In [23]:
print(output2.head())
print('\n{}'.format(output2['SalePrice'].min()))
print('\n{}'.format(output2['SalePrice'].max()))

     Id      SalePrice
0  1461  118770.721261
1  1462  146653.655137
2  1463  191059.481575
3  1464  187218.649312
4  1465  194365.513974

61616.60991538978

546838.5798302649
